# NutriScan Baseline V2 Retraining

Notebook ini menjalankan ulang baseline EfficientNet-B0 v2 di Google Colab GPU untuk dataset `nutriscan-mvp-food-dataset-v0.2`.

Sebelum menjalankan notebook:

1. Buka `Runtime > Change runtime type`.
2. Pilih `T4 GPU`.
3. Upload ZIP dataset ke Google Drive, misalnya `MyDrive/nutriscan-mvp-food-dataset-v0.2.zip`.

ZIP dataset harus berisi struktur berikut:

```txt
data/processed-v0.2/
  train/<class>/
  val/<class>/ atau validation/<class>/
  test/<class>/
```

## 1. Verify GPU

Jika `cuda_available` bernilai `False`, ubah runtime Colab ke GPU dulu sebelum lanjut training.

In [ ]:
import torch

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
print(f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

## 2. Configure Paths

Ubah `DATASET_ZIP` jika file ZIP dataset berada di folder Drive lain. Ubah `BRANCH` hanya kalau script belum ada di branch yang kamu checkout.

In [ ]:
REPO_URL = "https://github.com/BimoAtaullahR/nutri-scan.git"
BRANCH = "ai/baseline"
PROJECT_DIR = "/content/nutri-scan"
AI_DIR = f"{PROJECT_DIR}/services/ai-inference"
DATASET_ZIP = "/content/drive/MyDrive/nutriscan-mvp-food-dataset-v0.2.zip"
RESULTS_DIR = "/content/drive/MyDrive/nutriscan-baseline-v2-results"

## 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Clone Repo And Checkout Branch

Cell ini aman dijalankan ulang. Jika folder repo sudah ada, repo akan di-fetch dan checkout branch target.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, cwd=cwd, check=True)

if not Path(PROJECT_DIR).exists():
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

run("git fetch origin --prune", cwd=PROJECT_DIR)
run(f"git checkout {BRANCH}", cwd=PROJECT_DIR)
run(f"git pull --ff-only origin {BRANCH}", cwd=PROJECT_DIR)
run("git status --short --branch", cwd=PROJECT_DIR)

## 5. Extract Dataset

Dataset diextract ke folder `services/ai-inference`, sehingga path akhirnya menjadi `data/processed-v0.2`.

In [ ]:
from pathlib import Path

dataset_zip = Path(DATASET_ZIP)
if not dataset_zip.exists():
    raise FileNotFoundError(f"Dataset ZIP not found: {dataset_zip}")

run(f"unzip -q -o '{DATASET_ZIP}' -d '{AI_DIR}'")
run("find data/processed-v0.2 -maxdepth 2 -type d | sort | head -60", cwd=AI_DIR)

## 6. Sanity Check Dataset Counts

In [ ]:
from pathlib import Path

processed_dir = Path(AI_DIR) / "data/processed-v0.2"
if not processed_dir.exists():
    raise FileNotFoundError(f"Processed dataset folder not found: {processed_dir}")

for split_name in ["train", "val", "validation", "test"]:
    split_dir = processed_dir / split_name
    if not split_dir.exists():
        continue
    print(f"\n{split_name}")
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        count = sum(1 for path in class_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
        print(f"  {class_dir.name}: {count}")

## 7. Retrain Baseline V2

Cell ini menjalankan script helper:

- install dependencies
- cek CUDA
- dry-run config
- train EfficientNet-B0 v2
- evaluate predictions
- export misclassified images
- print summary top-1, top-3, dan weak classes

In [ ]:
run("REQUIRE_CUDA=1 INSTALL_DEPS=1 bash scripts/colab_retrain_baseline_v2.sh", cwd=AI_DIR)

## 8. Inspect Metrics

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

report_dir = Path(AI_DIR) / "reports/baseline-food-classifier-v2"
metrics = json.loads((report_dir / "metrics.json").read_text())
per_class = json.loads((report_dir / "per_class_metrics.json").read_text())

baseline_v1 = {
    "dataset": "v0.1 guideline baseline",
    "top1_accuracy": 0.8129,
    "top3_accuracy": 0.9698,
    "num_test_samples": 497,
    "meets_mvp_target": True,
}
baseline_v2 = {
    "dataset": "v0.2 reviewed retrain",
    "top1_accuracy": metrics["top1_accuracy"],
    "top3_accuracy": metrics["top3_accuracy"],
    "num_test_samples": metrics["num_test_samples"],
    "meets_mvp_target": metrics["meets_mvp_target"],
}

comparison = {
    "baseline_v1": baseline_v1,
    "baseline_v2": baseline_v2,
    "delta": {
        "top1_accuracy": baseline_v2["top1_accuracy"] - baseline_v1["top1_accuracy"],
        "top3_accuracy": baseline_v2["top3_accuracy"] - baseline_v1["top3_accuracy"],
        "num_test_samples": baseline_v2["num_test_samples"] - baseline_v1["num_test_samples"],
    },
    "weak_classes_v2": {
        label: per_class[label]
        for label in ["rendang", "gado_gado", "soto"]
    },
}

def pct(value):
    return f"{value * 100:.2f}%"

def signed_pct(value):
    sign = "+" if value >= 0 else ""
    return f"{sign}{value * 100:.2f} pp"

comparison_md = f"""
## Baseline Comparison

| Dataset | Top-1 | Top-3 | Test Samples | MVP Passed |
|---|---:|---:|---:|---|
| v0.1 guideline baseline | {pct(baseline_v1['top1_accuracy'])} | {pct(baseline_v1['top3_accuracy'])} | {baseline_v1['num_test_samples']} | {baseline_v1['meets_mvp_target']} |
| v0.2 reviewed retrain | {pct(baseline_v2['top1_accuracy'])} | {pct(baseline_v2['top3_accuracy'])} | {baseline_v2['num_test_samples']} | {baseline_v2['meets_mvp_target']} |
| Delta v0.2 - v0.1 | {signed_pct(comparison['delta']['top1_accuracy'])} | {signed_pct(comparison['delta']['top3_accuracy'])} | {comparison['delta']['num_test_samples']:+d} | - |

## Weak Classes, v0.2 Reviewed Retrain

| Class | Precision | Recall | F1 | Support |
|---|---:|---:|---:|---:|
"""

for label, values in comparison["weak_classes_v2"].items():
    comparison_md += (
        f"| {label} | {pct(values['precision'])} | {pct(values['recall'])} | "
        f"{pct(values['f1'])} | {values['support']} |\n"
    )

display(Markdown(comparison_md))
(report_dir / "baseline_comparison.md").write_text(comparison_md.strip() + "\n")
(report_dir / "baseline_comparison.json").write_text(json.dumps(comparison, indent=2) + "\n")

## 9. Save Results Back To Google Drive

Output ini yang perlu kamu ambil untuk review dan sharing. Jangan commit model artifact, dataset, atau generated reports ke Git.

In [ ]:
run(f"mkdir -p '{RESULTS_DIR}'")
run(f"cp -r model-artifacts/baseline-food-classifier-v2 '{RESULTS_DIR}/'", cwd=AI_DIR)
run(f"cp -r reports/baseline-food-classifier-v2 '{RESULTS_DIR}/'", cwd=AI_DIR)
run(f"find '{RESULTS_DIR}' -maxdepth 3 -type f | sort | head -80")

## Output Penting

Hasil utama setelah training:

```txt
model-artifacts/baseline-food-classifier-v2/model.pt
model-artifacts/baseline-food-classifier-v2/label_map.json
reports/baseline-food-classifier-v2/predictions.json
reports/baseline-food-classifier-v2/metrics.json
reports/baseline-food-classifier-v2/per_class_metrics.json
reports/baseline-food-classifier-v2/confusion_matrix.json
reports/baseline-food-classifier-v2/baseline_comparison.md
reports/baseline-food-classifier-v2/baseline_comparison.json
reports/baseline-food-classifier-v2/misclassified/
```

Cell `Inspect Metrics` otomatis membandingkan hasil retraining v0.2 dengan baseline guideline v0.1:

- v0.1 top-1 accuracy: `81.29%`
- v0.1 top-3 accuracy: `96.98%`
- v0.1 test samples: `497`
- weak classes: `rendang`, `gado_gado`, `soto`